<a href="https://colab.research.google.com/github/pi-sen/Order-Flow-Imbalance/blob/main/BlockhouseAssignment_Piyush.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Importing libraries
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Loading csv to dataframe
df = pd.read_csv('/content/drive/My Drive/Blockhouse/first_25000_rows.csv')

# Set display options to show all rows
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect display width

In [3]:
df.head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,bid_px_01,ask_px_01,bid_sz_01,ask_sz_01,bid_ct_01,ask_ct_01,bid_px_02,ask_px_02,bid_sz_02,ask_sz_02,bid_ct_02,ask_ct_02,bid_px_03,ask_px_03,bid_sz_03,ask_sz_03,bid_ct_03,ask_ct_03,bid_px_04,ask_px_04,bid_sz_04,ask_sz_04,bid_ct_04,ask_ct_04,bid_px_05,ask_px_05,bid_sz_05,ask_sz_05,bid_ct_05,ask_ct_05,bid_px_06,ask_px_06,bid_sz_06,ask_sz_06,bid_ct_06,ask_ct_06,bid_px_07,ask_px_07,bid_sz_07,ask_sz_07,bid_ct_07,ask_ct_07,bid_px_08,ask_px_08,bid_sz_08,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,130,166627,11405847,233.67,233.74,139,200,1,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.4,233.8,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.3,234.0,100,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,130,166814,11405848,233.67,233.74,141,200,2,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.4,233.8,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.3,234.0,100,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,130,166409,11405849,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.4,233.8,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.3,234.0,100,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,130,166400,11406449,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,400,8,2,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.4,233.8,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.3,234.0,100,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,130,166056,11406501,233.67,233.74,144,200,3,1,233.53,233.75,10,1,1,1,233.52,233.77,200,8,1,1,233.48,233.78,28,150,1,2,233.45,233.79,15,29,1,1,233.4,233.8,110,25,2,1,233.39,233.94,10,200,1,1,233.31,233.98,110,44,2,3,233.3,234.0,100,155,1,7,233.25,234.13,55,400,2,1,AAPL


In [25]:
class OrderFlowImbalance:
    def __init__(self, data_path=None, df=None):
        """
        Initialize the OrderFlowImbalance calculator with either a data path or a DataFrame.

        Parameters:
        data_path : str, optional
            Path to the CSV file containing order book data.
        df : pandas.DataFrame, optional
            DataFrame containing order book data.
        """
        if df is not None:
            self.df = df.copy()
        elif data_path is not None:
            self.df = pd.read_csv(data_path)
        else:
            raise ValueError("Either data_path or df must be provided")

        # Prepare the data
        self._prepare_data()

        # Define column names for different levels
        self.price_bid_cols = [f'bid_px_{i:02}' for i in range(10)]
        self.size_bid_cols = [f'bid_sz_{i:02}' for i in range(10)]
        self.size_ask_cols = [f'ask_sz_{i:02}' for i in range(10)]
        self.price_ask_cols = [f'ask_px_{i:02}' for i in range(10)]

        self.cross_asset_results = None

    def _prepare_data(self):
        """Prepare the data for OFI calculation."""
        # Use ts_event to reflect actual market event timing
        if 'ts_event' in self.df.columns:
            self.df['ts_event'] = pd.to_datetime(self.df['ts_event'])
            # Sort chronologically by event time and symbol
            self.df = self.df.sort_values(['symbol','ts_event']).reset_index(drop=True)

        # Calculate mid price for return calculations (needed for cross-asset OFI)
        if all(col in self.df.columns for col in ['bid_px_00', 'ask_px_00']):
            self.df['mid_px'] = (self.df['bid_px_00'] + self.df['ask_px_00']) / 2

    def computebestofi(self,row_t, row_t_minus_1):
      """
      Compute the best level Order Flow Imbalance

      Parameters:
      row_t : pandas.Series
          Current row data
      row_tm1 : pandas.Series
          Previous row data

      Returns:

      The OFI value : float
        """
      # Extract the bid ask prices and sizes
      pb_t, qb_t = row_t[self.price_bid_cols[0]], row_t[self.size_bid_cols[0]]
      pa_t, qa_t = row_t[self.price_ask_cols[0]], row_t[self.size_ask_cols[0]]
      pb_tminus1, qb_tminus1 = row_t_minus_1[self.price_bid_cols[0]], row_t_minus_1[self.size_bid_cols[0]]
      pa_tminus1, qa_tminus1 = row_t_minus_1[self.price_ask_cols[0]], row_t_minus_1[self.size_ask_cols[0]]

      # Calculate the OFI for bids and asks
      if pb_t > pb_tminus1:
        ofi_bid = qb_t
      elif pb_t == pb_tminus1:
        ofi_bid = qb_t - qb_tminus1
      else:
        ofi_bid = -qb_t

      if pa_t > pa_tminus1:
        ofi_ask = -qa_t
      elif pa_t == pa_tminus1:
        ofi_ask = qa_t - qa_tminus1
      else:
        ofi_ask = qa_t

      return ofi_bid - ofi_ask

    def calculate_best_ofi(self):
      """
      Calculate the best level OFI for each row in the DataFrame.

      Returns:
      pandas.DataFrame
          DataFrame with the best level OFI values.
      """
      # Initialize OFI best column with zeros
      self.df['Best_OFI'] = 0.0

      # Iterate over each row in the DataFrame for different symbols
      for symbol in self.df['symbol'].unique():
        symbol_mask = self.df['symbol'] == symbol
        symbol_indices = self.df[symbol_mask].index

      # Skip the first row of each symbol
      for i in range(1,len(symbol_indices)):
        curr_idx = symbol_indices[i]
        prev_idx = symbol_indices[i-1]

        # Calculate the best OFI between consecutive rows of the same symbol
        self.df.loc[curr_idx,'Best_OFI'] = self.computebestofi(self.df.loc[curr_idx], self.df.loc[prev_idx])

      return self.df

    def compute_ofi_for_level(self, row_t, row_tm1, level):
      """
      Compute OFI for a specific level.

      Parameters:
      row_t : pandas.Series
          Current row data
      row_tm1 : pandas.Series
          Previous row data
      level : int
          The level index (0-9) to compute OFI for

      Returns:
      float
          The OFI value for the specified level
      """
      # Extract bid/ask prices and sizes for the specified level
      pb_t = row_t[self.price_bid_cols[level]]
      qb_t = row_t[self.size_bid_cols[level]]
      pb_tm1 = row_tm1[self.price_bid_cols[level]]
      qb_tm1 = row_tm1[self.size_bid_cols[level]]
      pa_t = row_t[self.price_ask_cols[level]]
      qa_t = row_t[self.size_ask_cols[level]]
      pa_tm1 = row_tm1[self.price_ask_cols[level]]
      qa_tm1 = row_tm1[self.size_ask_cols[level]]

      # Bid logic
      if pb_t > pb_tm1:
          ofi_bid = qb_t
      elif pb_t == pb_tm1:
          ofi_bid = qb_t - qb_tm1
      else:
          ofi_bid = -qb_t

      # Ask logic
      if pa_t > pa_tm1:
          ofi_ask = -qa_t
      elif pa_t == pa_tm1:
          ofi_ask = qa_t - qa_tm1
      else:
          ofi_ask = qa_t

      return ofi_bid - ofi_ask

    def calculate_average_depth(self, symbol_indices):
        """
        Calculate the average order book depth across all 10 levels.

        Parameters:
        symbol_indices : array-like
            Indices of rows belonging to a specific symbol

        Returns:
        float
            Average depth across all levels
        """
        # Number of events
        delta_N = len(symbol_indices)

        # Skip if not enough data
        if delta_N < 2:
            return 1.0  # Default value to avoid division by zero

        # Sum of all bid and ask sizes across all levels and events
        total_size = 0

        for idx in symbol_indices:
            row = self.df.loc[idx]
            for level in range(10):
                # Add bid and ask sizes for this level
                total_size += row[self.size_bid_cols[level]] + row[self.size_ask_cols[level]]

        # Calculate average depth per formula: Q_i,t^M,h = (1/M) * (1/(2*ΔN(t))) * sum[q_i,n^m,b + q_i,n^m,a]
        avg_depth = (1/10) * (1/(2*delta_N)) * total_size

        return max(avg_depth, 1.0)  # Ensure we don't divide by zero later

    def calculate_multi_level_ofi(self):
        """
        Calculate multi-level OFI

        Returns:
        pandas.DataFrame
            The original DataFrame with added multi-level OFI columns
        """
        # Initialize columns for each level's OFI
        for level in range(10):
            self.df[f'OFI_level_{level}'] = 0.0

        # Initialize column for normalized multi-level OFI
        self.df['Multi_Level_OFI'] = 0.0

        # Process each symbol separately
        for symbol in self.df['symbol'].unique():
            symbol_mask = self.df['symbol'] == symbol
            symbol_indices = self.df[symbol_mask].index

            # Calculate average depth for this symbol
            avg_depth = self.calculate_average_depth(symbol_indices)

        # Skip first row for each symbol
        for i in range(1, len(symbol_indices)):
            curr_idx = symbol_indices[i]
            prev_idx = symbol_indices[i-1]

            # Calculate normalized OFI for each level
            for level in range(10):
                # Calculate raw OFI for this level
                ofi_value = self.compute_ofi_for_level(
                    self.df.loc[curr_idx],
                    self.df.loc[prev_idx],
                    level
                )

                # Normalize by the average depth
                if avg_depth > 0:
                    normalized_ofi = ofi_value / avg_depth
                else:
                    normalized_ofi = 0.0


                # Store the normalized OFI
                self.df.loc[curr_idx, f'Normalized_OFI_level_{level}'] = normalized_ofi

        # Create the vector column
        self.df['Multi_Level_OFI_Vector'] = self.df.apply(
        lambda row: np.array([
            row[f'Normalized_OFI_level_{level}'] if not pd.isna(row[f'Normalized_OFI_level_{level}'])
            else 0.0  # Replace NaN with 0.0
            for level in range(10)
        ]),
        axis=1
        )

        return self.df

    def calculate_integrated_ofi(self):
        """
        Calculate Integrated OFI using PCA.
        This method takes the multi-level OFI vectors, applies PCA,
        and retains only the first principal component normalized by its L1 norm.

        Returns:

        pandas.DataFrame
            The original DataFrame with added Integrated_OFI column
        """

        # Ensure presence of multi-level OFI first
        if 'Multi_Level_OFI_Vector' not in self.df.columns:
            self.calculate_multi_level_ofi()

        # Initialize Integrated OFI column
        self.df['Integrated_OFI'] = 0.0

        # Create arrays for vectorized operations
        level_columns = [f'Normalized_OFI_level_{level}' for level in range(10)]

        # Process each symbol separately
        for symbol in self.df['symbol'].unique():
            symbol_mask = self.df['symbol'] == symbol
            symbol_indices = self.df[symbol_mask].index

            # Extract valid multi-level OFI vectors (non-zero)
            valid_indices = []
            ofi_vectors = []

            for idx in symbol_indices:
                vector = self.df.loc[idx, 'Multi_Level_OFI_Vector']
                if np.any(vector != 0):
                    valid_indices.append(idx)
                    ofi_vectors.append(vector)

            # Skip if we don't have enough data
            if len(ofi_vectors) < 2:
                continue

            # Extract vectors directly from columns
            ofi_vectors_df = self.df.loc[valid_indices, level_columns]

            # Handle NaN values - use an imputer to replace NaNs with the mean
            imputer = SimpleImputer(strategy='mean')
            ofi_vectors = imputer.fit_transform(ofi_vectors_df)

            # Make sure we still have enough data after handling NaNs
            if ofi_vectors.shape[0] < 2 or np.all(ofi_vectors == 0):
                continue

            # Apply PCA to extract the first principal component
            pca = PCA(n_components=1)
            pca.fit(ofi_vectors)

            # Get the first principal vector
            w1 = pca.components_[0]

            # Normalize w1 by its L1 norm (sum of absolute values)
            w1_norm = np.sum(np.abs(w1))
            w1_normalized = w1 / w1_norm if w1_norm > 0 else w1

            # Store the normalized weights for reference
            weight_dict = {f'PCA_weight_level_{i}': weight for i, weight in enumerate(w1_normalized)}
            for key, value in weight_dict.items():
                # Store the weight value in the first valid row for this symbol
                if valid_indices:
                    self.df.loc[valid_indices[0], key] = value

            # Calculate integrated OFI for each valid row
            for idx in valid_indices:
                vector = self.df.loc[idx, 'Multi_Level_OFI_Vector']
                # Apply formula: ofi^I,h = (w1^T * ofi^(h)) / ||w1||_1
                # Since w1 is already normalized, we just need the dot product
                integrated_ofi = np.dot(w1_normalized, vector)
                self.df.loc[idx, 'Integrated_OFI'] = integrated_ofi

        return self.df

    def calculate_cross_asset_ofi(self):
        """
        Calculate cross-asset OFI using LASSO regression to identify significant
        cross-impacts between assets. Implements both best-level and integrated
        cross-asset OFI models.

        Returns:
        dict
            A dictionary with results for both best-level and integrated cross-impact models
        """
        # Check if we have multiple symbols to analyze
        symbols = self.df['symbol'].unique()
        if len(symbols) < 2:
            print("Cross-asset OFI requires at least 2 different symbols.")
            return None

        # Prepare data containers for results
        best_level_results = {}
        integrated_results = {}

        # For each target symbol, build a model using other symbols' OFI
        for target_symbol in symbols:
            # Get data for target symbol (the dependent variable)
            target_df = self.df[self.df['symbol'] == target_symbol].copy()

            # Create lagged price to calculate returns
            target_df['price_lag'] = target_df['mid_px'].shift(1)

            # Calculate returns (as percentage change)
            target_df['returns'] = (target_df['mid_px'] - target_df['price_lag']) / target_df['price_lag']

            # Drop rows with NaN returns (first row)
            target_df = target_df.dropna(subset=['returns']).copy()

            if len(target_df) < 10:  # Need sufficient data for regression
                continue

            # Create feature matrices for both models
            X_best = pd.DataFrame(index=target_df.index)
            X_integrated = pd.DataFrame(index=target_df.index)

            # Add target symbol's own OFI as "Self" feature
            X_best[f'{target_symbol}_Best_OFI'] = target_df['Best_OFI']
            X_integrated[f'{target_symbol}_Integrated_OFI'] = target_df['Integrated_OFI']

            # Add other symbols' OFI as "Cross" features
            for other_symbol in symbols:
                if other_symbol == target_symbol:
                    continue

                # Get best-level OFI for other symbol
                other_df = self.df[self.df['symbol'] == other_symbol]

                # Align timestamps with target symbol
                joined_best = pd.merge_asof(
                    target_df[['ts_event', 'returns']],
                    other_df[['ts_event', 'Best_OFI']],
                    on='ts_event',
                    direction='nearest',
                    tolerance=pd.Timedelta('1min')  # Tolerance
                )

                # Add to feature matrix if we have matching data
                if not joined_best['Best_OFI'].isnull().all():
                    X_best[f'{other_symbol}_Best_OFI'] = joined_best['Best_OFI'].values

                # Do the same for integrated OFI
                joined_integrated = pd.merge_asof(
                    target_df[['ts_event', 'returns']],
                    other_df[['ts_event', 'Integrated_OFI']],
                    on='ts_event',
                    direction='nearest',
                    tolerance=pd.Timedelta('1min')
                )

                if not joined_integrated['Integrated_OFI'].isnull().all():
                    X_integrated[f'{other_symbol}_Integrated_OFI'] = joined_integrated['Integrated_OFI'].values

            # Prepare response variable (returns)
            y = target_df['returns'].values

            # Standardize features
            scaler_best = StandardScaler()
            scaler_integrated = StandardScaler()

            X_best_scaled = scaler_best.fit_transform(X_best.fillna(0))
            X_integrated_scaled = scaler_integrated.fit_transform(X_integrated.fillna(0))

            # Apply LASSO regression for best-level cross-impact (CI^[1])
            alpha_best = 1e-10   # Adjust alpha as needed for desired sparsity
            lasso_best = Lasso(alpha=alpha_best, max_iter=10000, fit_intercept=True)
            lasso_best.fit(X_best_scaled, y)

            # Store best-level results
            coeffs_best = {}
            for i, feature in enumerate(X_best.columns):
                coeffs_best[feature] = lasso_best.coef_[i]

            best_level_results[target_symbol] = {
                'intercept': lasso_best.intercept_,
                'coefficients': coeffs_best,
                'model_r2': lasso_best.score(X_best_scaled, y),
                'num_nonzero_cross': sum(1 for f, c in coeffs_best.items()
                                      if not f.startswith(f'{target_symbol}_') and abs(c) > 1e-6)
            }

            # Apply LASSO regression for integrated cross-impact (CI')
            alpha_integrated = 1e-10   # Adjust as needed
            lasso_integrated = Lasso(alpha=alpha_integrated, max_iter=10000, fit_intercept=True)
            lasso_integrated.fit(X_integrated_scaled, y)

            # Store integrated results
            coeffs_integrated = {}
            for i, feature in enumerate(X_integrated.columns):
                coeffs_integrated[feature] = lasso_integrated.coef_[i]

            integrated_results[target_symbol] = {
                'intercept': lasso_integrated.intercept_,
                'coefficients': coeffs_integrated,
                'model_r2': lasso_integrated.score(X_integrated_scaled, y),
                'num_nonzero_cross': sum(1 for f, c in coeffs_integrated.items()
                                      if not f.startswith(f'{target_symbol}_') and abs(c) > 1e-6)
            }

        # Create summary of cross-impact
        cross_impact_summary = {
            'best_level': self._create_cross_impact_matrix(best_level_results, symbols),
            'integrated': self._create_cross_impact_matrix(integrated_results, symbols)
        }

        return {
            'best_level_models': best_level_results,
            'integrated_models': integrated_results,
            'cross_impact_matrix': cross_impact_summary
        }

    def _create_cross_impact_matrix(self, results, symbols):
        """
        Create a cross-impact matrix from LASSO regression results.

        Parameters:

        results : dict
            Dictionary of regression results
        symbols : list
            List of symbols

        Returns:

        pandas.DataFrame
            Matrix showing cross-impact between symbols
        """

        # Initialize matrix with zeros
        matrix = pd.DataFrame(0.0, index=symbols, columns=symbols)

        # Fill with coefficients
        for target in results:
            coeffs = results[target]['coefficients']
            for feature, value in coeffs.items():
                # Extract source symbol from feature name (format: "SYMBOL_OFI_TYPE")
                source_symbol = feature.split('_')[0]
                if source_symbol in symbols and source_symbol != target:
                    matrix.loc[target, source_symbol] = value

        return matrix

    def calculate_all_ofi_features(self):
        """
        Calculate all OFI features: Best-Level and Multi-Level.

        Returns:

        pandas.DataFrame
            The DataFrame with all OFI features added
        """
        self.calculate_best_ofi()
        self.calculate_multi_level_ofi()
        self.calculate_integrated_ofi()

        # Only calculate cross-asset OFI if we have multiple symbols
        if 'symbol' in self.df.columns and len(self.df['symbol'].unique()) > 1:
            print("Calculating cross-asset OFI...")
            self.cross_asset_results = self.calculate_cross_asset_ofi()

            if self.cross_asset_results is None:
                print("Cross-asset OFI calculation failed.")
        else:
            print("Skipping cross-asset OFI (insufficient symbols).")

        return self.df

    def create_synthetic_multi_asset_data(self, num_synthetic_symbols=5):
        """
        Create a synthetic multi-asset dataframe by adding modified copies of the original data
        with different symbols and strong, realistic correlations.

        Parameters:

        original_df : pandas.DataFrame
            The original dataframe with a single symbol
        num_synthetic_symbols : int
            Number of additional synthetic symbols to create

        Returns:

        pandas.DataFrame
            Combined dataframe with original and synthetic data
        """

        # Store the original symbol
        original_symbol = self.df['symbol'].iloc[0]
        print(f"Creating synthetic data based on original symbol: {original_symbol}")

        # Create a list to hold all dataframes
        all_dfs = [self.df.copy()]

        # Define synthetic symbols
        synthetic_symbols = ['MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META'][:num_synthetic_symbols]

        # Define correlation patterns
        correlation_patterns = {
            'MSFT': {
                'time_window': [(9, 12), (14, 16)],  # 9am-12pm and 2pm-4pm stronger correlation
                'strength': 0.7,                     # 70% correlation during these windows
                'base_strength': 0.3                 # 30% correlation at other times
            },
            'GOOGL': {
                'time_window': [(10, 11), (15, 16)], # 10am-11am and 3pm-4pm stronger correlation
                'strength': 0.6,
                'base_strength': 0.2
            },
            'AMZN': {
                'time_window': [(9, 10), (15, 16)],
                'strength': 0.8,
                'base_strength': 0.4
            },
            'TSLA': {
                'time_window': [(9, 9.5), (15.5, 16)],
                'strength': 0.5,
                'base_strength': 0.2
            },
            'META': {
                'time_window': [(10, 12), (14, 15)],
                'strength': 0.6,
                'base_strength': 0.25
            }
        }

        # Create price patterns with correlations
        # Get the original price pattern from bid/ask
        original_prices = (self.df['bid_px_00'] + self.df['ask_px_00']) / 2
        original_returns = original_prices.pct_change().fillna(0)

        # For each synthetic symbol
        for symbol in synthetic_symbols:
            # Create a copy of the original dataframe
            synthetic_df = self.df.copy()

            # Change the symbol
            synthetic_df['symbol'] = symbol

            # Create correlated price changes based on patterns
            corr_pattern = correlation_patterns.get(symbol, {'strength': 0.5, 'base_strength': 0.2, 'time_window': []})

            # Extract timestamps to determine correlation strength at each point
            timestamp_hours = pd.to_datetime(synthetic_df['ts_event']).dt.hour + pd.to_datetime(synthetic_df['ts_event']).dt.minute/60

            # Create correlation mask based on time windows
            correlation_strength = np.ones(len(synthetic_df)) * corr_pattern['base_strength']
            for start, end in corr_pattern.get('time_window', []):
                in_window = (timestamp_hours >= start) & (timestamp_hours <= end)
                correlation_strength[in_window] = corr_pattern['strength']

            # Adjust prices based on symbol and correlation pattern
            price_factors = {
                'MSFT': 1.5,    # MSFT is ~1.5x AAPL's price
                'GOOGL': 5.0,   # GOOGL is ~5x AAPL's price
                'AMZN': 2.0,    # AMZN is ~2x AAPL's price
                'TSLA': 0.8,    # TSLA is ~0.8x AAPL's price
                'META': 1.2     # META is ~1.2x AAPL's price
            }

            price_factor = price_factors.get(symbol, 1.2)

            # Generate correlated synthetic returns
            synthetic_returns = np.zeros(len(original_returns))
            for i in range(1, len(synthetic_returns)):
                # Correlation component
                corr_component = original_returns.iloc[i] * correlation_strength[i]

                # Independent component
                indep_component = np.random.normal(0, 0.001) * (1 - correlation_strength[i])

                # Combined return (with a small offset to create lead/lag effects)
                synthetic_returns[i] = corr_component + indep_component

            # Convert returns to price levels
            synthetic_prices = (1 + synthetic_returns).cumprod() * original_prices.iloc[0] * price_factor

            # Apply to all price columns
            for level in range(10):
                bid_col = f'bid_px_{level:02}'
                ask_col = f'ask_px_{level:02}'

                # Calculate price differentials from original
                bid_diff = self.df[bid_col] - original_prices
                ask_diff = self.df[ask_col] - original_prices

                # Apply to synthetic data
                synthetic_df[bid_col] = synthetic_prices + bid_diff
                synthetic_df[ask_col] = synthetic_prices + ask_diff

            # Create correlated order book depths
            for level in range(10):
                bid_sz_col = f'bid_sz_{level:02}'
                ask_sz_col = f'ask_sz_{level:02}'

                for col in [bid_sz_col, ask_sz_col]:
                    if col in synthetic_df.columns:
                        # Base sizes on original with correlation plus some independent variation
                        orig_sizes = self.df[col].values

                        # Create correlated size variation
                        correlated_sizes = np.zeros(len(orig_sizes))
                        for i in range(len(correlated_sizes)):
                            # Correlation component + independent component
                            correlated_sizes[i] = (
                                orig_sizes[i] * correlation_strength[i] +
                                orig_sizes[i] * (1 - correlation_strength[i]) * np.random.uniform(0.7, 1.3)
                            )

                        # Set the correlated sizes
                        synthetic_df[col] = correlated_sizes.astype(int)

            # Add time shift to make the data more realistic
            time_shift = pd.Timedelta(seconds=np.random.randint(-30, 30))
            synthetic_df['ts_event'] = pd.to_datetime(synthetic_df['ts_event']) + time_shift
            synthetic_df['ts_recv'] = pd.to_datetime(synthetic_df['ts_recv']) + time_shift

            # Add this synthetic dataframe to our list
            all_dfs.append(synthetic_df)

        # Combine all dataframes
        combined_df = pd.concat(all_dfs, ignore_index=True)

        # Sort by timestamp to ensure chronological order
        combined_df = combined_df.sort_values('ts_event').reset_index(drop=True)

        # Calculate mid price for returns calculations
        combined_df['mid_px'] = (combined_df['bid_px_00'] + combined_df['ask_px_00']) / 2

        print(f"Created synthetic data with symbols: {combined_df['symbol'].unique()}")
        return combined_df


In [28]:
# Main function
if __name__ == "__main__":
  # Create an instance with the data path
  ofi = OrderFlowImbalance(data_path='/content/drive/My Drive/Blockhouse/first_25000_rows.csv')

  # Compute the best level OFI
  result_df = ofi.calculate_all_ofi_features()

Skipping cross-asset OFI (insufficient symbols).


In [29]:
# Check the respective columns
columns = ['Best_OFI', 'Multi_Level_OFI_Vector', 'Integrated_OFI']
display(result_df[columns].head(30))

,Best_OFI,Multi_Level_OFI_Vector,Integrated_OFI
0,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.00000000
1,2.0,"[0.01024806045207884, 0.0, 0.0, 0.0, 0.0, 0.0,...",0.00029475
2,3.0,"[0.01537209067811826, 0.0, 0.0, 0.0, 0.0, 0.0,...",0.00044213
3,0.0,"[0.0, 0.0, 1.024806045207884, 0.0, 0.0, 0.0, 0...",0.14481400
4,0.0,"[0.0, 0.0, -1.024806045207884, 0.0, 0.0, 0.0, ...",-0.14481400
5,0.0,"[0.0, 2.049612090415768, 0.051240302260394194,...",0.16868404
6,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.024806045207...",-0.25884181
7,-200.0,"[-1.024806045207884, 0.0, 0.0, 0.0, 0.0, 0.0, ...",-0.02947549
8,1.0,"[0.00512403022603942, 0.04099224180831536, 0.7...",0.59689121
9,-200.0,"[-1.024806045207884, -0.00512403022603942, -0....",-0.45675143


In [27]:
# Main function for testing
if __name__ == "__main__":

    # Load the data
    data_path = '/content/drive/My Drive/Blockhouse/first_25000_rows.csv'
    original_df = pd.read_csv(data_path)

    # Convert timestamps to datetime if needed
    if 'ts_event' in original_df.columns and original_df['ts_event'].dtype != 'datetime64[ns]':
        original_df['ts_event'] = pd.to_datetime(original_df['ts_event'])

    # Create OFI calculator instance
    ofi = OrderFlowImbalance(df=original_df)

    # Check how many symbols we have
    symbols = original_df['symbol'].unique()
    print(f"Original data has {len(symbols)} symbols: {symbols}")

    # If only one symbol, create synthetic data
    if len(symbols) < 2:
        print("Only one symbol found. Creating synthetic multi-asset data for testing.")
        multi_asset_df = ofi.create_synthetic_multi_asset_data(num_synthetic_symbols=5)

        # Create a new OFI calculator with the multi-asset data
        ofi = OrderFlowImbalance(df=multi_asset_df)

    # Calculate all OFI features
    result_df = ofi.calculate_all_ofi_features()

    if hasattr(ofi, 'cross_asset_results') and ofi.cross_asset_results is not None:
        print("\nCross-Asset OFI Results:")

        # Format the matrix
        best_level_matrix = ofi.cross_asset_results['cross_impact_matrix']['best_level']
        integrated_matrix = ofi.cross_asset_results['cross_impact_matrix']['integrated']

        print("\nBest-Level Cross-Impact Matrix:")
        pd.set_option('display.precision', 8)
        print(best_level_matrix)

        print("\nIntegrated Cross-Impact Matrix:")
        print(integrated_matrix)


Original data has 1 symbols: ['AAPL']
Only one symbol found. Creating synthetic multi-asset data for testing...
Creating synthetic data based on original symbol: AAPL
Created synthetic data with symbols: ['TSLA' 'META' 'MSFT' 'GOOGL' 'AAPL' 'AMZN']
Calculating OFI features...
Calculating cross-asset OFI...

Cross-Asset OFI Results:

Best-Level Cross-Impact Matrix:
       AAPL  AMZN  GOOGL  META  MSFT        TSLA
AAPL    0.0   0.0    0.0   0.0   0.0  0.00000028
AMZN    0.0   0.0    0.0   0.0   0.0 -0.00000084
GOOGL   0.0   0.0    0.0   0.0   0.0  0.00001152
META    0.0   0.0    0.0   0.0   0.0  0.00001304
MSFT    0.0   0.0    0.0   0.0   0.0 -0.00000373
TSLA    0.0   0.0    0.0   0.0   0.0  0.00000000

Integrated Cross-Impact Matrix:
       AAPL  AMZN  GOOGL  META  MSFT        TSLA
AAPL    0.0   0.0    0.0   0.0   0.0 -0.00000015
AMZN    0.0   0.0    0.0   0.0   0.0  0.00000581
GOOGL   0.0   0.0    0.0   0.0   0.0  0.00000655
META    0.0   0.0    0.0   0.0   0.0  0.00001887
MSFT    0.0 